In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import httpx
from langchain_groq import ChatGroq

# Disable SSL verification to work around corporate proxy/firewall
client = httpx.Client(verify=False)
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, http_client=client)

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [4]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

I apologize for the confusion earlier. Here are some recipe suggestions using leftover chicken and rice:

You can make a Chicken Fried Rice dish, which is a popular Chinese-inspired recipe. 

You can also make Chicken and Rice Cakes.

If you're looking for something more specific, I can try to search again. 

Would you like me to provide the recipe instructions for any of these dishes?


In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='58c50440-1c50-4cdb-8b1c-c88dbcf5f240'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dbgs5jcbx', 'function': {'arguments': '{"query":"recipes using leftover chicken and rice"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 776, 'total_tokens': 810, 'completion_time': 0.081985412, 'completion_tokens_details': None, 'prompt_time': 0.031017212, 'prompt_tokens_details': None, 'queue_time': 0.047178142, 'total_time': 0.113002624}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc3f8-7436-7a10-9a7e-4060b24a5651-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes